# Vignette 5: Low-data benchmark

The paper's central result (Figure 3): with 16 to 64 real images per class, adding 10 synthetic
images per real image raised nine-way accuracy by up to 13 points; with 228 per class the gain
disappeared. This notebook runs that sweep at small scale on the six-label sample, with and
without synthetic augmentation, over several seeds, and plots accuracy with Wald confidence
intervals. It is the template the full-scale experiments reuse.

In [ ]:
SMALL = True
SYNTHETIC_DIR = None
N_REAL_LIST = [8, 32] if SMALL else [8, 16, 32, 56]
SEEDS = [0] if SMALL else [0, 1, 2]
N_SYNTHETIC_PER_REAL = 10
MAX_EPOCHS = 8 if SMALL else 50
BACKBONE = "tf_efficientnetv2_m.in1k"

In [ ]:
import os, json, time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image

os.environ.setdefault("HF_HOME", "/workspace/.cache/huggingface")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # less fragmentation on a 32 GB card
WORK = Path(os.environ.get("SYNDERM_WORK_DIR", "/workspace/synderm"))
WORK.mkdir(parents=True, exist_ok=True)
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
from synderm.data import load_sample, load_2024_shards, SyntheticDataset
from synderm.data.atlas import HELD_OUT_CSV
from synderm.utils import synthetic_train_val_split
from synderm.classify import ClassificationDataset, build_transforms, train_classifier
real_train, real_val = load_sample(split="train"), load_sample(split="val")
classes = real_train.labels
# Synthetic images derived from validation or held-out real images are excluded (no leakage).
banned = set(real_val.df["md5hash"]) | set(pd.read_csv(HELD_OUT_CSV)["md5hash"])
POLICY = "ref-in-train" if SYNTHETIC_DIR else "ref-not-in-val"
synthetic = SyntheticDataset(SYNTHETIC_DIR, labels=classes) if SYNTHETIC_DIR else load_2024_shards(
    classes, method="finetune-text-to-image", submethod="text-to-image", max_per_label=N_SYNTHETIC_PER_REAL * 64,
    exclude_ref_ids=banned)
records = []
for n_real in N_REAL_LIST:
    for with_syn in (False, True):
        for seed in SEEDS:
            train, val = synthetic_train_val_split(real_train, synthetic if with_syn else None, per_class_test_size=8,
                                                   n_real_per_class=n_real, n_synthetic_per_real=N_SYNTHETIC_PER_REAL, random_state=seed, synthetic_policy=POLICY, exclude_ref_ids=banned)
            res = train_classifier(ClassificationDataset(train, classes, build_transforms(augment=True)),
                                   ClassificationDataset(val, classes), classes,
                                   test_dataset=ClassificationDataset(real_val, classes), backbone=BACKBONE,
                                   max_epochs=MAX_EPOCHS, seed=seed, verbose=False)
            m = res.test_metrics
            records.append({"n_real": n_real, "synthetic": with_syn, "seed": seed, "n_train": len(train),
                            "accuracy": m["accuracy"], "correct": m["correct"], "n_test": m["n"]})
            print(records[-1])
df = pd.DataFrame(records)
(WORK / "vignettes").mkdir(parents=True, exist_ok=True)
df.to_csv(WORK / "vignettes" / "05_low_data_benchmark.csv", index=False)
df

In [ ]:
from synderm.classify import accuracy_ci
agg = df.groupby(["n_real", "synthetic"]).agg(correct=("correct", "sum"), n=("n_test", "sum")).reset_index()
agg[["acc", "lo", "hi"]] = agg.apply(lambda r: pd.Series(accuracy_ci(int(r.correct), int(r.n))), axis=1)
fig, ax = plt.subplots(figsize=(5, 3.5))
for with_syn, grp in agg.groupby("synthetic"):
    ax.errorbar(grp.n_real, grp.acc, yerr=[grp.acc - grp.lo, grp.hi - grp.acc], marker="o", capsize=3,
                label="real + 10 synthetic per real" if with_syn else "real only")
ax.set_xscale("log", base=2); ax.set_xticks(N_REAL_LIST); ax.set_xticklabels(N_REAL_LIST)
ax.set_xlabel("real images per class"); ax.set_ylabel("held-out accuracy (mean over seeds, Wald 95% CI)")
ax.legend(); plt.tight_layout(); plt.show()
agg.round(3)

The paper's Figure 3 used 1 to 228 real images per class, five seeds and a 360-image test set;
scale `N_REAL_LIST`, `SEEDS` and the sample size accordingly for a full run. Pooling correct
counts across seeds (as above) gives one confidence interval per condition; the paper also
reports Benjamini-Hochberg adjusted two-sample proportion tests between the two curves.